# AegisAML: 01 - Data Pipeline & Feature Engineering

This notebook handles data loading, validation, EDA, and complex AML feature engineering.


In [ ]:
"""
Project Configuration and Imports
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from typing import Dict, Any, List
import warnings
warnings.filterwarnings("ignore")


In [ ]:
def setup_logger() -> logging.Logger:
    """Configures the logger for the pipeline."""
    logger = logging.getLogger("AegisAML_Data")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        ch = logging.StreamHandler()
        ch.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
        logger.addHandler(ch)
    return logger

logger = setup_logger()


## Dataset Loading


In [ ]:
def load_dataset(filepath: str) -> pd.DataFrame:
    """Loads dataset with error handling."""
    try:
        df = pd.read_csv(filepath)
        logger.info(f"Loaded dataset from {filepath} with shape {df.shape}")
        return df
    except Exception as e:
        logger.error(f"Failed to load dataset: {e}")
        raise

transactions = load_dataset("../data/transactions.csv")


## Feature Engineering: Velocity and Rolling Stats


In [ ]:
def calculate_velocity_features(df: pd.DataFrame) -> pd.DataFrame:
    """Calculates transaction velocity (count/sum over time windows)."""
    logger.info("Calculating velocity features...")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(by=["sender_account", "timestamp"])
    
    # 24h rolling count
    df["txn_count_24h"] = df.groupby("sender_account")["timestamp"].transform(lambda x: x.diff().dt.total_seconds() < 86400).astype(int)
    
    logger.info("Velocity features generated.")
    return df

transactions = calculate_velocity_features(transactions)
